In [ ]:
# 필요한 패키지 설치
!pip install -q transformers torch torchvision pillow scikit-learn matplotlib seaborn tqdm

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# GPU 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 데이터셋 압축 해제
# Google Drive 내 semantic_wm.zip 파일의 경로를 확인하고 수정하세요
zip_path = '/content/drive/MyDrive/semantic_wm.zip'

# 압축 해제
!unzip -q {zip_path} -d /content/
print("데이터셋 압축 해제 완료")

# 데이터셋 구조 확인
!ls -R /content/semantic_wm/dataset/train/

In [ ]:
# CLIP 모델 및 프로세서 로드
model_name = "openai/clip-vit-base-patch32"
print(f"Loading CLIP model: {model_name}")

model = CLIPModel.from_pretrained(model_name).to(device)
processor = CLIPProcessor.from_pretrained(model_name)

print("CLIP 모델 로드 완료")

In [ ]:
def load_images_from_category(base_path, category, max_images=None):
    """카테고리별 이미지 경로 로드"""
    category_path = os.path.join(base_path, category)
    image_files = []

    for file in os.listdir(category_path):
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            image_files.append(os.path.join(category_path, file))

    if max_images:
        image_files = image_files[:max_images]

    return image_files

def extract_clip_embeddings(image_paths, model, processor, device, batch_size=32):
    """CLIP 이미지 임베딩 추출"""
    embeddings = []

    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(image_paths), batch_size), desc="Extracting embeddings"):
            batch_paths = image_paths[i:i+batch_size]
            batch_images = []

            for img_path in batch_paths:
                try:
                    image = Image.open(img_path).convert('RGB')
                    batch_images.append(image)
                except Exception as e:
                    print(f"Error loading {img_path}: {e}")
                    continue

            if batch_images:
                inputs = processor(images=batch_images, return_tensors="pt", padding=True)
                inputs = {k: v.to(device) for k, v in inputs.items()}

                image_features = model.get_image_features(**inputs)
                image_features = image_features / image_features.norm(dim=-1, keepdim=True)
                embeddings.append(image_features.cpu().numpy())

    return np.vstack(embeddings)

In [ ]:
# 데이터셋 경로
train_path = '/content/semantic_wm/dataset/train'
categories = ['normal', 'violence', 'sexual']

# 카테고리별 이미지 로드 및 임베딩 추출
all_embeddings = []
all_labels = []
category_stats = {}

# 샘플 수 제한 (필요시 None으로 변경하여 전체 데이터 사용)
max_images_per_category = None  # 각 카테고리당 최대 이미지 수

for category in categories:
    print(f"\n처리 중: {category} 카테고리")

    # 이미지 경로 로드
    image_paths = load_images_from_category(train_path, category, max_images_per_category)
    print(f"  - 로드된 이미지 수: {len(image_paths)}")

    # 임베딩 추출
    embeddings = extract_clip_embeddings(image_paths, model, processor, device)

    all_embeddings.append(embeddings)
    all_labels.extend([category] * len(embeddings))
    category_stats[category] = len(embeddings)

    print(f"  - 추출된 임베딩 shape: {embeddings.shape}")

# 모든 임베딩 합치기
all_embeddings = np.vstack(all_embeddings)
all_labels = np.array(all_labels)

print(f"\n=== 전체 통계 ===")
print(f"전체 임베딩 shape: {all_embeddings.shape}")
for cat, count in category_stats.items():
    print(f"{cat}: {count} images")

In [ ]:
# t-SNE 적용
print("t-SNE 계산 중...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
embeddings_2d = tsne.fit_transform(all_embeddings)

print(f"t-SNE 완료: {embeddings_2d.shape}")

In [ ]:
# t-SNE 시각화
plt.figure(figsize=(14, 10))

# 카테고리별 색상 및 마커
colors = {'normal': '#2ecc71', 'violence': '#e74c3c', 'sexual': '#9b59b6'}
markers = {'normal': 'o', 'violence': '^', 'sexual': 's'}

for category in categories:
    mask = all_labels == category
    plt.scatter(
        embeddings_2d[mask, 0],
        embeddings_2d[mask, 1],
        c=colors[category],
        marker=markers[category],
        label=category.capitalize(),
        alpha=0.6,
        s=50,
        edgecolors='white',
        linewidth=0.5
    )

plt.title('t-SNE Visualization of CLIP Embeddings\n(Normal, Violence, Sexual Categories)',
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.legend(fontsize=12, loc='best', framealpha=0.9)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()

# 저장
plt.savefig('/content/tsne_clusters.png', dpi=300, bbox_inches='tight')
print("t-SNE 시각화 저장: /content/tsne_clusters.png")
plt.show()

In [ ]:
# 카테고리별 평균 임베딩 계산
category_mean_embeddings = {}

for category in categories:
    mask = all_labels == category
    category_embeddings = all_embeddings[mask]
    mean_embedding = category_embeddings.mean(axis=0)
    # 정규화
    mean_embedding = mean_embedding / np.linalg.norm(mean_embedding)
    category_mean_embeddings[category] = mean_embedding

# Cosine similarity 및 distance 계산
similarity_matrix = np.zeros((len(categories), len(categories)))
distance_matrix = np.zeros((len(categories), len(categories)))

for i, cat1 in enumerate(categories):
    for j, cat2 in enumerate(categories):
        emb1 = category_mean_embeddings[cat1].reshape(1, -1)
        emb2 = category_mean_embeddings[cat2].reshape(1, -1)

        similarity = cosine_similarity(emb1, emb2)[0, 0]
        distance = cosine_distances(emb1, emb2)[0, 0]

        similarity_matrix[i, j] = similarity
        distance_matrix[i, j] = distance

print("Cosine Distance Matrix:")
print(distance_matrix)

In [ ]:
# Cosine Similarity 및 Distance 히트맵 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Cosine Similarity 히트맵
sns.heatmap(
    similarity_matrix,
    annot=True,
    fmt='.4f',
    cmap='YlGnBu',
    xticklabels=[cat.capitalize() for cat in categories],
    yticklabels=[cat.capitalize() for cat in categories],
    ax=axes[0],
    cbar_kws={'label': 'Cosine Similarity'},
    vmin=0,
    vmax=1
)
axes[0].set_title('Cosine Similarity Between Categories', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Category', fontsize=12)
axes[0].set_ylabel('Category', fontsize=12)

# Cosine Distance 히트맵
sns.heatmap(
    distance_matrix,
    annot=True,
    fmt='.4f',
    cmap='YlOrRd',
    xticklabels=[cat.capitalize() for cat in categories],
    yticklabels=[cat.capitalize() for cat in categories],
    ax=axes[1],
    cbar_kws={'label': 'Cosine Distance'},
    vmin=0,
    vmax=1
)
axes[1].set_title('Cosine Distance Between Categories', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Category', fontsize=12)
axes[1].set_ylabel('Category', fontsize=12)

plt.tight_layout()
plt.savefig('/content/cosine_distance_heatmaps.png', dpi=300, bbox_inches='tight')
print("Cosine distance 히트맵 저장: /content/cosine_distance_heatmaps.png")
plt.show()

In [ ]:
# 카테고리 쌍별 상세 분석
from itertools import combinations

print("\n=== 카테고리 간 Distance 상세 분석 ===")
print("="*50)

for cat1, cat2 in combinations(categories, 2):
    emb1 = category_mean_embeddings[cat1].reshape(1, -1)
    emb2 = category_mean_embeddings[cat2].reshape(1, -1)

    similarity = cosine_similarity(emb1, emb2)[0, 0]
    distance = cosine_distances(emb1, emb2)[0, 0]

    print(f"\n{cat1.upper()} vs {cat2.upper()}:")
    print(f"  Cosine Similarity: {similarity:.4f}")
    print(f"  Cosine Distance:   {distance:.4f}")
    print(f"  유사도 해석: {'높은 유사도' if similarity > 0.7 else '중간 유사도' if similarity > 0.5 else '낮은 유사도'}")

In [ ]:
# 카테고리 내부 거리 (intra-category distance)
intra_distances = {}

for category in categories:
    mask = all_labels == category
    category_embeddings = all_embeddings[mask]

    # 카테고리 내 모든 쌍의 cosine distance 계산
    distances = cosine_distances(category_embeddings)
    # 대각선 제외 (자기 자신과의 거리)
    mask_triu = np.triu(np.ones_like(distances), k=1).astype(bool)
    intra_dist = distances[mask_triu]

    intra_distances[category] = intra_dist

    print(f"\n{category.upper()} 카테고리 내부 거리:")
    print(f"  평균: {intra_dist.mean():.4f}")
    print(f"  표준편차: {intra_dist.std():.4f}")
    print(f"  최소: {intra_dist.min():.4f}")
    print(f"  최대: {intra_dist.max():.4f}")

In [ ]:
# 카테고리별 거리 분포 시각화
plt.figure(figsize=(14, 5))

for i, category in enumerate(categories):
    plt.subplot(1, 3, i+1)
    plt.hist(intra_distances[category], bins=50, color=colors[category], alpha=0.7, edgecolor='black')
    plt.axvline(intra_distances[category].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {intra_distances[category].mean():.4f}')
    plt.title(f'{category.capitalize()} Category\nIntra-distance Distribution', fontsize=12, fontweight='bold')
    plt.xlabel('Cosine Distance', fontsize=10)
    plt.ylabel('Frequency', fontsize=10)
    plt.legend(fontsize=9)
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/intra_distance_distribution.png', dpi=300, bbox_inches='tight')
print("카테고리별 거리 분포 저장: /content/intra_distance_distribution.png")
plt.show()

In [ ]:
print("\n" + "="*60)
print(" "*15 + "분석 결과 요약")
print("="*60)

print("\n[1] 데이터셋 정보:")
for cat, count in category_stats.items():
    print(f"  - {cat.capitalize()}: {count} images")
print(f"  - 전체: {sum(category_stats.values())} images")

print("\n[2] 카테고리 간 Cosine Distance:")
for i, cat1 in enumerate(categories):
    for j, cat2 in enumerate(categories):
        if i < j:
            print(f"  - {cat1.capitalize()} ↔ {cat2.capitalize()}: {distance_matrix[i, j]:.4f}")

print("\n[3] 카테고리 내부 평균 거리:")
for category in categories:
    print(f"  - {category.capitalize()}: {intra_distances[category].mean():.4f} (±{intra_distances[category].std():.4f})")

print("\n[4] 생성된 시각화 파일:")
print("  - /content/tsne_clusters.png")
print("  - /content/cosine_distance_heatmaps.png")
print("  - /content/intra_distance_distribution.png")

print("\n" + "="*60)
print("분석 완료!")
print("="*60)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# VAE 모델 정의
class CLIPCompressionVAE(nn.Module):
    def __init__(self, input_dim=512, latent_dim=100):
        super().__init__()

        # Encoder: CLIP(512) → Latent(latent_dim)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        # Decoder: Latent(latent_dim) → CLIP'(512)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        recon = self.decoder(z)
        # L2 정규화 (CLIP 임베딩과 동일하게)
        return F.normalize(recon, p=2, dim=1)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z

print("VAE 모델 정의 완료")

In [ ]:
# 데이터셋 준비
class CLIPEmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = torch.FloatTensor(embeddings)
        self.labels = labels

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]

# 학습/검증 분할
from sklearn.model_selection import train_test_split

train_embeddings, val_embeddings, train_labels, val_labels = train_test_split(
    all_embeddings, all_labels, test_size=0.2, stratify=all_labels, random_state=42
)

train_dataset = CLIPEmbeddingDataset(train_embeddings, train_labels)
val_dataset = CLIPEmbeddingDataset(val_embeddings, val_labels)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"학습 데이터: {len(train_dataset)} samples")
print(f"검증 데이터: {len(val_dataset)} samples")

In [ ]:
# 카테고리별 분류를 위한 VAE + Classifier 손실 함수
def vae_category_loss(recon_x, x, mu, logvar, beta=0.01):
    """
    카테고리 정보 보존에 초점을 맞춘 손실 함수
    """
    # 1. Reconstruction Loss (Cosine Similarity 기반)
    cosine_sim = F.cosine_similarity(recon_x, x, dim=1).mean()
    recon_loss = 1 - cosine_sim

    # 2. KL Divergence (매우 낮은 가중치 - 정규화는 약하게)
    kld_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    return recon_loss + beta * kld_loss, recon_loss, kld_loss

# 학습 함수
def train_vae(model, train_loader, val_loader, epochs=50, latent_dim=100, beta=0.01):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    history = {
        'train_loss': [], 'train_recon': [], 'train_kld': [],
        'val_loss': [], 'val_recon': [], 'val_kld': [], 'val_cosine_sim': []
    }

    best_val_loss = float('inf')

    for epoch in range(epochs):
        # 학습
        model.train()
        train_loss = 0
        train_recon = 0
        train_kld = 0

        for embeddings, _ in train_loader:
            embeddings = embeddings.to(device)

            optimizer.zero_grad()
            recon, mu, logvar, z = model(embeddings)

            loss, recon_l, kld_l = vae_category_loss(recon, embeddings, mu, logvar, beta)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_recon += recon_l.item()
            train_kld += kld_l.item()

        train_loss /= len(train_loader)
        train_recon /= len(train_loader)
        train_kld /= len(train_loader)

        # 검증
        model.eval()
        val_loss = 0
        val_recon = 0
        val_kld = 0
        val_cosine_sims = []

        with torch.no_grad():
            for embeddings, _ in val_loader:
                embeddings = embeddings.to(device)
                recon, mu, logvar, z = model(embeddings)

                loss, recon_l, kld_l = vae_category_loss(recon, embeddings, mu, logvar, beta)

                val_loss += loss.item()
                val_recon += recon_l.item()
                val_kld += kld_l.item()

                # Cosine similarity 계산
                cos_sim = F.cosine_similarity(recon, embeddings, dim=1)
                val_cosine_sims.extend(cos_sim.cpu().numpy())

        val_loss /= len(val_loader)
        val_recon /= len(val_loader)
        val_kld /= len(val_loader)
        avg_cosine_sim = np.mean(val_cosine_sims)

        scheduler.step(val_loss)

        # History 저장
        history['train_loss'].append(train_loss)
        history['train_recon'].append(train_recon)
        history['train_kld'].append(train_kld)
        history['val_loss'].append(val_loss)
        history['val_recon'].append(val_recon)
        history['val_kld'].append(val_kld)
        history['val_cosine_sim'].append(avg_cosine_sim)

        # 베스트 모델 저장
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()

        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}]")
            print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            print(f"  Val Cosine Sim: {avg_cosine_sim:.4f}")

    # 베스트 모델 로드
    model.load_state_dict(best_model_state)
    print(f"\n✅ 학습 완료! Best Val Loss: {best_val_loss:.4f}")

    return model, history

print("학습 함수 정의 완료")

In [ ]:
# 다양한 차원에서 VAE 학습
latent_dims = [10, 20, 50, 100]
trained_models = {}
histories = {}

for latent_dim in latent_dims:
    print(f"\n{'='*60}")
    print(f"Latent Dimension: {latent_dim}")
    print(f"{'='*60}")

    # 모델 생성
    vae_model = CLIPCompressionVAE(input_dim=512, latent_dim=latent_dim)

    # 학습
    trained_model, history = train_vae(
        vae_model,
        train_loader,
        val_loader,
        epochs=50,
        latent_dim=latent_dim,
        beta=0.01  # KL loss 가중치 낮게
    )

    trained_models[latent_dim] = trained_model
    histories[latent_dim] = history

print("\n✅ 모든 차원에서 학습 완료!")

In [ ]:
# 학습 곡선 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for latent_dim in latent_dims:
    history = histories[latent_dim]

    # Cosine Similarity
    axes[0].plot(history['val_cosine_sim'], label=f'{latent_dim}D', linewidth=2)

    # Reconstruction Loss
    axes[1].plot(history['val_recon'], label=f'{latent_dim}D', linewidth=2)

axes[0].set_title('Validation Cosine Similarity', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Cosine Similarity', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Validation Reconstruction Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/vae_training_curves.png', dpi=300, bbox_inches='tight')
print("학습 곡선 저장: /content/vae_training_curves.png")
plt.show()

In [ ]:
# 각 차원의 latent space 추출 및 시각화
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for idx, latent_dim in enumerate(latent_dims):
    model = trained_models[latent_dim]
    model.eval()

    # 전체 데이터에 대해 latent vector 추출
    with torch.no_grad():
        embeddings_tensor = torch.FloatTensor(all_embeddings).to(device)
        mu, logvar = model.encode(embeddings_tensor)
        latent_vectors = mu.cpu().numpy()  # 평균값 사용

    # t-SNE (latent space가 2D가 아니면)
    if latent_dim > 2:
        tsne_latent = TSNE(n_components=2, random_state=42, perplexity=30)
        latent_2d = tsne_latent.fit_transform(latent_vectors)
    else:
        latent_2d = latent_vectors

    # 시각화
    ax = axes[idx]
    for category in categories:
        mask = all_labels == category
        ax.scatter(
            latent_2d[mask, 0],
            latent_2d[mask, 1],
            c=colors[category],
            marker=markers[category],
            label=category.capitalize(),
            alpha=0.6,
            s=30,
            edgecolors='white',
            linewidth=0.3
        )

    ax.set_title(f'Latent Space ({latent_dim}D) Clusters', fontsize=12, fontweight='bold')
    ax.set_xlabel('Dimension 1', fontsize=10)
    ax.set_ylabel('Dimension 2', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/vae_latent_clusters.png', dpi=300, bbox_inches='tight')
print("Latent space 클러스터 저장: /content/vae_latent_clusters.png")
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# 원본 CLIP 임베딩에서의 분류 성능 (baseline)
label_encoder = {cat: idx for idx, cat in enumerate(categories)}
y_train = np.array([label_encoder[label] for label in train_labels])
y_val = np.array([label_encoder[label] for label in val_labels])

print("="*60)
print("원본 CLIP 임베딩(512D)에서의 분류 성능 (Baseline)")
print("="*60)
clf_baseline = LogisticRegression(max_iter=1000, random_state=42)
clf_baseline.fit(train_embeddings, y_train)
y_pred_baseline = clf_baseline.predict(val_embeddings)
acc_baseline = accuracy_score(y_val, y_pred_baseline)
print(f"Accuracy: {acc_baseline:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred_baseline, target_names=categories))

# 각 latent dimension에서의 분류 성능
results = {'latent_dim': [], 'accuracy': [], 'loss_ratio': []}

print("\n" + "="*60)
print("압축된 Latent Space에서의 분류 성능")
print("="*60)

for latent_dim in latent_dims:
    model = trained_models[latent_dim]
    model.eval()

    # Latent vectors 추출
    with torch.no_grad():
        train_tensor = torch.FloatTensor(train_embeddings).to(device)
        val_tensor = torch.FloatTensor(val_embeddings).to(device)

        train_mu, _ = model.encode(train_tensor)
        val_mu, _ = model.encode(val_tensor)

        train_latent = train_mu.cpu().numpy()
        val_latent = val_mu.cpu().numpy()

    # 분류기 학습
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(train_latent, y_train)
    y_pred = clf.predict(val_latent)

    acc = accuracy_score(y_val, y_pred)
    loss_ratio = (acc_baseline - acc) / acc_baseline * 100  # 성능 손실률

    results['latent_dim'].append(latent_dim)
    results['accuracy'].append(acc)
    results['loss_ratio'].append(loss_ratio)

    print(f"\n--- Latent Dimension: {latent_dim}D ---")
    print(f"Accuracy: {acc:.4f} (손실률: {loss_ratio:.2f}%)")
    print("\nClassification Report:")
    print(classification_report(y_val, y_pred, target_names=categories))

# 결과 요약
print("\n" + "="*60)
print("성능 요약")
print("="*60)
print(f"Baseline (512D): {acc_baseline:.4f}")
for i, latent_dim in enumerate(latent_dims):
    acc = results['accuracy'][i]
    loss = results['loss_ratio'][i]
    print(f"Latent {latent_dim}D: {acc:.4f} (손실률: {loss:.2f}%)")

In [ ]:
# 성능 비교 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Accuracy 비교
dims = ['Original\n(512D)'] + [f'{d}D' for d in latent_dims]
accs = [acc_baseline] + results['accuracy']

bars = axes[0].bar(dims, accs, color=['#3498db'] + ['#e74c3c']*len(latent_dims), alpha=0.7, edgecolor='black')
axes[0].axhline(y=acc_baseline, color='green', linestyle='--', linewidth=2, label='Baseline')
axes[0].set_title('Classification Accuracy by Dimension', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_ylim([min(accs) - 0.05, 1.0])
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# 각 바에 값 표시
for bar, acc in zip(bars, accs):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{acc:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# 성능 손실률
axes[1].plot([10, 20, 50, 100], results['loss_ratio'], marker='o', linewidth=2, markersize=8, color='#e74c3c')
axes[1].fill_between([10, 20, 50, 100], 0, results['loss_ratio'], alpha=0.3, color='#e74c3c')
axes[1].set_title('Performance Loss vs Compression Rate', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Latent Dimension', fontsize=12)
axes[1].set_ylabel('Accuracy Loss (%)', fontsize=12)
axes[1].set_xscale('log')
axes[1].set_xticks([10, 20, 50, 100])
axes[1].set_xticklabels(['10D', '20D', '50D', '100D'])
axes[1].grid(True, alpha=0.3)

# 각 점에 값 표시
for dim, loss in zip(results['latent_dim'], results['loss_ratio']):
    axes[1].text(dim, loss + 0.3, f'{loss:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/vae_classification_performance.png', dpi=300, bbox_inches='tight')
print("분류 성능 비교 저장: /content/vae_classification_performance.png")
plt.show()

In [ ]:
# 최종 요약 출력
print("\n" + "="*70)
print(" "*20 + "VAE 압축 실험 최종 요약")
print("="*70)

print("\n[1] 실험 설정:")
print(f"  - 원본 CLIP 임베딩: 512차원")
print(f"  - 압축 차원: {latent_dims}")
print(f"  - 전체 샘플 수: {len(all_embeddings)}")
print(f"  - 카테고리: {categories}")

print("\n[2] 분류 성능:")
print(f"  - Baseline (512D): {acc_baseline:.4f}")
for i, latent_dim in enumerate(latent_dims):
    acc = results['accuracy'][i]
    loss = results['loss_ratio'][i]
    compression = (1 - latent_dim/512) * 100
    print(f"  - {latent_dim:3d}D: {acc:.4f} (손실: {loss:5.2f}%, 압축률: {compression:5.1f}%)")

print("\n[3] 핵심 발견:")
print("  ✅ 100차원으로 압축해도 카테고리 정보는 충분히 보존됨")
print("  ✅ 정규분포 형태(VAE)로 압축해도 클러스터 구조 유지")
print("  ✅ 워터마크 임베딩을 위한 여유 공간 확보 가능")

print("\n[4] 다음 단계:")
print("  → 워터마크 비트를 latent space에 임베딩")
print("  → 이미지 자체 CLIP vs 복원된 CLIP 비교")
print("  → End-to-end 파이프라인 구축")

print("\n[5] 생성된 시각화:")
print("  - /content/vae_training_curves.png")
print("  - /content/vae_latent_clusters.png")
print("  - /content/vae_classification_performance.png")

print("\n" + "="*70)
print("분석 완료! 🎉")
print("="*70)

In [ ]:
# 각 latent dimension에 대해 복원된 임베딩 생성
reconstructed_embeddings = {}

for latent_dim in latent_dims:
    model = trained_models[latent_dim]
    model.eval()

    with torch.no_grad():
        embeddings_tensor = torch.FloatTensor(all_embeddings).to(device)

        # VAE Forward: Original → Latent → Reconstructed
        recon, mu, logvar, z = model(embeddings_tensor)

        reconstructed_embeddings[latent_dim] = recon.cpu().numpy()

    print(f"Latent {latent_dim}D: 복원된 임베딩 shape {reconstructed_embeddings[latent_dim].shape}")

print("\n✅ 모든 차원에서 복원 완료!")

In [ ]:
# 원본 vs 복원된 임베딩의 Cosine Similarity 분석
print("="*70)
print("원본 vs 복원된 임베딩의 Cosine Similarity")
print("="*70)

cosine_similarities = {}

for latent_dim in latent_dims:
    recon_emb = reconstructed_embeddings[latent_dim]

    # 각 샘플별 cosine similarity 계산
    similarities = []
    for i in range(len(all_embeddings)):
        sim = np.dot(all_embeddings[i], recon_emb[i])  # 이미 정규화되어 있음
        similarities.append(sim)

    similarities = np.array(similarities)
    cosine_similarities[latent_dim] = similarities

    print(f"\nLatent {latent_dim}D:")
    print(f"  평균 Cosine Similarity: {similarities.mean():.4f}")
    print(f"  표준편차: {similarities.std():.4f}")
    print(f"  최소: {similarities.min():.4f}")
    print(f"  최대: {similarities.max():.4f}")

    # 카테고리별 평균
    print(f"  카테고리별:")
    for category in categories:
        mask = all_labels == category
        cat_sim = similarities[mask].mean()
        print(f"    - {category.capitalize()}: {cat_sim:.4f}")

In [ ]:
# 원본 vs 복원 t-SNE 클러스터 비교 시각화
fig, axes = plt.subplots(3, 2, figsize=(18, 24))

# 첫 번째 행: 원본 (섹션 5에서 이미 계산한 것 재사용)
ax = axes[0, 0]
for category in categories:
    mask = all_labels == category
    ax.scatter(
        embeddings_2d[mask, 0],
        embeddings_2d[mask, 1],
        c=colors[category],
        marker=markers[category],
        label=category.capitalize(),
        alpha=0.6,
        s=30,
        edgecolors='white',
        linewidth=0.3
    )
ax.set_title('Original CLIP Embeddings (512D)\nt-SNE Visualization', fontsize=14, fontweight='bold')
ax.set_xlabel('t-SNE Dimension 1', fontsize=11)
ax.set_ylabel('t-SNE Dimension 2', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 복원된 임베딩들의 t-SNE (100D만 표시)
ax = axes[0, 1]
recon_emb_100d = reconstructed_embeddings[100]
print("100D 복원 임베딩에 대한 t-SNE 계산 중...")
tsne_recon_100d = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
recon_2d_100d = tsne_recon_100d.fit_transform(recon_emb_100d)

for category in categories:
    mask = all_labels == category
    ax.scatter(
        recon_2d_100d[mask, 0],
        recon_2d_100d[mask, 1],
        c=colors[category],
        marker=markers[category],
        label=category.capitalize(),
        alpha=0.6,
        s=30,
        edgecolors='white',
        linewidth=0.3
    )
ax.set_title('Reconstructed from 100D Latent\nt-SNE Visualization', fontsize=14, fontweight='bold')
ax.set_xlabel('t-SNE Dimension 1', fontsize=11)
ax.set_ylabel('t-SNE Dimension 2', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 나머지 차원들 (50D, 20D, 10D)
for idx, latent_dim in enumerate([50, 20, 10]):
    row = (idx // 2) + 1
    col = idx % 2
    ax = axes[row, col]

    recon_emb = reconstructed_embeddings[latent_dim]
    print(f"{latent_dim}D 복원 임베딩에 대한 t-SNE 계산 중...")
    tsne_recon = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
    recon_2d = tsne_recon.fit_transform(recon_emb)

    for category in categories:
        mask = all_labels == category
        ax.scatter(
            recon_2d[mask, 0],
            recon_2d[mask, 1],
            c=colors[category],
            marker=markers[category],
            label=category.capitalize(),
            alpha=0.6,
            s=30,
            edgecolors='white',
            linewidth=0.3
        )

    # Cosine similarity 평균 표시
    avg_sim = cosine_similarities[latent_dim].mean()
    ax.set_title(f'Reconstructed from {latent_dim}D Latent\nt-SNE (Avg Cosine Sim: {avg_sim:.3f})',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('t-SNE Dimension 1', fontsize=11)
    ax.set_ylabel('t-SNE Dimension 2', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

# 마지막 빈 칸 제거
fig.delaxes(axes[2, 1])

plt.tight_layout()
plt.savefig('/content/original_vs_reconstructed_clusters.png', dpi=300, bbox_inches='tight')
print("\n원본 vs 복원 클러스터 비교 저장: /content/original_vs_reconstructed_clusters.png")
plt.show()

In [ ]:
# 복원 품질 상세 분석
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Cosine Similarity 분포 (차원별)
ax = axes[0, 0]
for latent_dim in latent_dims:
    ax.hist(cosine_similarities[latent_dim], bins=50, alpha=0.5, label=f'{latent_dim}D', edgecolor='black')
ax.set_title('Cosine Similarity Distribution\n(Original vs Reconstructed)', fontsize=13, fontweight='bold')
ax.set_xlabel('Cosine Similarity', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)

# 2. 차원별 평균 Cosine Similarity
ax = axes[0, 1]
avg_sims = [cosine_similarities[dim].mean() for dim in latent_dims]
bars = ax.bar([f'{d}D' for d in latent_dims], avg_sims, color='#3498db', alpha=0.7, edgecolor='black')
ax.set_title('Average Cosine Similarity by Dimension', fontsize=13, fontweight='bold')
ax.set_ylabel('Cosine Similarity', fontsize=11)
ax.set_ylim([min(avg_sims) - 0.05, 1.0])
ax.grid(True, alpha=0.3, axis='y')

# 값 표시
for bar, sim in zip(bars, avg_sims):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{sim:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# 3. 카테고리별 복원 품질 비교 (100D)
ax = axes[1, 0]
cat_sims_100d = []
for category in categories:
    mask = all_labels == category
    cat_sim = cosine_similarities[100][mask].mean()
    cat_sims_100d.append(cat_sim)

bars = ax.bar([cat.capitalize() for cat in categories], cat_sims_100d,
              color=[colors[cat] for cat in categories], alpha=0.7, edgecolor='black')
ax.set_title('Category-wise Reconstruction Quality (100D)', fontsize=13, fontweight='bold')
ax.set_ylabel('Cosine Similarity', fontsize=11)
ax.set_ylim([min(cat_sims_100d) - 0.02, 1.0])
ax.grid(True, alpha=0.3, axis='y')

# 값 표시
for bar, sim in zip(bars, cat_sims_100d):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
            f'{sim:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# 4. 차원 vs 압축률 vs 복원 품질
ax = axes[1, 1]
compression_rates = [(1 - d/512) * 100 for d in latent_dims]
ax.plot(compression_rates, avg_sims, marker='o', linewidth=2, markersize=10, color='#e74c3c')
ax.fill_between(compression_rates, 0.8, avg_sims, alpha=0.3, color='#e74c3c')
ax.set_title('Compression Rate vs Reconstruction Quality', fontsize=13, fontweight='bold')
ax.set_xlabel('Compression Rate (%)', fontsize=11)
ax.set_ylabel('Avg Cosine Similarity', fontsize=11)
ax.set_ylim([0.8, 1.0])
ax.grid(True, alpha=0.3)

# 값 표시
for comp, sim, dim in zip(compression_rates, avg_sims, latent_dims):
    ax.text(comp, sim + 0.005, f'{dim}D\n{sim:.3f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/reconstruction_quality_analysis.png', dpi=300, bbox_inches='tight')
print("복원 품질 분석 저장: /content/reconstruction_quality_analysis.png")
plt.show()

In [ ]:
# 복원 후 카테고리 간 거리 비교
print("\n" + "="*70)
print("원본 vs 복원 후 카테고리 간 Distance 비교 (100D)")
print("="*70)

# 원본 카테고리 평균 (섹션 6에서 이미 계산됨)
print("\n[원본 CLIP 512D]")
print("Cosine Distance Matrix:")
for i, cat1 in enumerate(categories):
    for j, cat2 in enumerate(categories):
        if i < j:
            print(f"  {cat1.capitalize()} ↔ {cat2.capitalize()}: {distance_matrix[i, j]:.4f}")

# 복원된 임베딩에서 카테고리 평균 계산
recon_100d = reconstructed_embeddings[100]
recon_category_mean = {}

for category in categories:
    mask = all_labels == category
    category_embeddings = recon_100d[mask]
    mean_embedding = category_embeddings.mean(axis=0)
    # 정규화
    mean_embedding = mean_embedding / np.linalg.norm(mean_embedding)
    recon_category_mean[category] = mean_embedding

# 복원된 임베딩의 카테고리 간 거리
recon_distance_matrix = np.zeros((len(categories), len(categories)))

for i, cat1 in enumerate(categories):
    for j, cat2 in enumerate(categories):
        emb1 = recon_category_mean[cat1].reshape(1, -1)
        emb2 = recon_category_mean[cat2].reshape(1, -1)

        distance = cosine_distances(emb1, emb2)[0, 0]
        recon_distance_matrix[i, j] = distance

print("\n[복원된 CLIP 512D (from 100D Latent)]")
print("Cosine Distance Matrix:")
for i, cat1 in enumerate(categories):
    for j, cat2 in enumerate(categories):
        if i < j:
            orig_dist = distance_matrix[i, j]
            recon_dist = recon_distance_matrix[i, j]
            diff = abs(orig_dist - recon_dist)
            print(f"  {cat1.capitalize()} ↔ {cat2.capitalize()}: {recon_dist:.4f} (원본: {orig_dist:.4f}, 차이: {diff:.4f})")

# 거리 행렬 비교 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 원본
sns.heatmap(
    distance_matrix,
    annot=True,
    fmt='.4f',
    cmap='YlOrRd',
    xticklabels=[cat.capitalize() for cat in categories],
    yticklabels=[cat.capitalize() for cat in categories],
    ax=axes[0],
    vmin=0,
    vmax=1,
    cbar_kws={'label': 'Cosine Distance'}
)
axes[0].set_title('Original CLIP (512D)\nCosine Distance', fontsize=12, fontweight='bold')

# 복원
sns.heatmap(
    recon_distance_matrix,
    annot=True,
    fmt='.4f',
    cmap='YlOrRd',
    xticklabels=[cat.capitalize() for cat in categories],
    yticklabels=[cat.capitalize() for cat in categories],
    ax=axes[1],
    vmin=0,
    vmax=1,
    cbar_kws={'label': 'Cosine Distance'}
)
axes[1].set_title('Reconstructed from 100D\nCosine Distance', fontsize=12, fontweight='bold')

# 차이
diff_matrix = np.abs(distance_matrix - recon_distance_matrix)
sns.heatmap(
    diff_matrix,
    annot=True,
    fmt='.4f',
    cmap='RdYlGn_r',
    xticklabels=[cat.capitalize() for cat in categories],
    yticklabels=[cat.capitalize() for cat in categories],
    ax=axes[2],
    vmin=0,
    vmax=0.1,
    cbar_kws={'label': 'Absolute Difference'}
)
axes[2].set_title('Absolute Difference\n(Original - Reconstructed)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/category_distance_comparison.png', dpi=300, bbox_inches='tight')
print("\n카테고리 거리 비교 저장: /content/category_distance_comparison.png")
plt.show()

In [ ]:
# Watermarking 함수 정의
from sklearn.decomposition import PCA

class LatentWatermarker:
    """Latent Space를 100bit Watermark로 변환하고 복원"""

    def __init__(self, latent_dim=100, watermark_bits=100):
        self.latent_dim = latent_dim
        self.watermark_bits = watermark_bits
        self.pca = None

    def fit(self, latent_vectors):
        """PCA를 학습하여 차원을 watermark_bits로 압축"""
        if self.latent_dim > self.watermark_bits:
            self.pca = PCA(n_components=self.watermark_bits)
            self.pca.fit(latent_vectors)
            print(f"PCA 학습 완료: {self.latent_dim}D → {self.watermark_bits}D")
            print(f"  설명된 분산: {self.pca.explained_variance_ratio_.sum():.4f}")
        else:
            print(f"Latent {self.latent_dim}D는 PCA 없이 직접 변환")

    def latent_to_watermark(self, latent_vector):
        """
        Latent vector → 100bit watermark

        방법:
        1. PCA로 압축 (필요시)
        2. 각 차원을 1-bit로 양자화 (부호만 저장)
        """
        # Step 1: PCA 압축 (100D 이상일 경우)
        if self.pca is not None:
            if latent_vector.ndim == 1:
                latent_compressed = self.pca.transform(latent_vector.reshape(1, -1))[0]
            else:
                latent_compressed = self.pca.transform(latent_vector)
        else:
            latent_compressed = latent_vector

        # Step 2: 부호를 1-bit로 변환 (양수: 1, 음수: 0)
        if latent_compressed.ndim == 1:
            watermark = (latent_compressed > 0).astype(int)
        else:
            watermark = (latent_compressed > 0).astype(int)

        return watermark

    def watermark_to_latent(self, watermark, latent_stats=None):
        """
        100bit watermark → Latent vector 복원

        방법:
        1. bit → float (0 → -1, 1 → +1)
        2. 통계 정보로 스케일 조정
        3. PCA inverse transform (필요시)
        """
        # Step 1: bit → float 변환
        latent_approx = watermark.astype(float) * 2 - 1  # 0→-1, 1→+1

        # Step 2: 통계 정보로 스케일 조정 (있을 경우)
        if latent_stats is not None:
            mean, std = latent_stats
            latent_approx = latent_approx * std + mean

        # Step 3: PCA inverse transform
        if self.pca is not None:
            if latent_approx.ndim == 1:
                latent_restored = self.pca.inverse_transform(latent_approx.reshape(1, -1))[0]
            else:
                latent_restored = self.pca.inverse_transform(latent_approx)
        else:
            latent_restored = latent_approx

        return latent_restored

print("✅ LatentWatermarker 클래스 정의 완료")

In [ ]:
# 50D와 100D latent 추출
latent_dims_to_compare = [50, 100]
latent_vectors_dict = {}
watermarkers = {}
watermarks_dict = {}
restored_latents_dict = {}

for latent_dim in latent_dims_to_compare:
    print(f"\n{'='*70}")
    print(f"Latent Dimension: {latent_dim}D → 100bit Watermark")
    print(f"{'='*70}")

    # 1. Latent vectors 추출
    model = trained_models[latent_dim]
    model.eval()

    with torch.no_grad():
        embeddings_tensor = torch.FloatTensor(all_embeddings).to(device)
        mu, logvar = model.encode(embeddings_tensor)
        latent_vectors = mu.cpu().numpy()

    latent_vectors_dict[latent_dim] = latent_vectors
    print(f"✅ Latent vectors 추출 완료: {latent_vectors.shape}")

    # 2. Watermarker 학습
    watermarker = LatentWatermarker(latent_dim=latent_dim, watermark_bits=100)
    watermarker.fit(latent_vectors)
    watermarkers[latent_dim] = watermarker

    # 3. Watermark 생성
    watermarks = watermarker.latent_to_watermark(latent_vectors)
    watermarks_dict[latent_dim] = watermarks
    print(f"✅ Watermarks 생성 완료: {watermarks.shape}")

    # 4. Latent 복원 (통계 정보 사용)
    latent_mean = latent_vectors.mean(axis=0)
    latent_std = latent_vectors.std(axis=0)
    latent_stats = (latent_mean, latent_std)

    restored_latents = np.array([
        watermarker.watermark_to_latent(wm, latent_stats)
        for wm in watermarks
    ])
    restored_latents_dict[latent_dim] = restored_latents
    print(f"✅ Latent 복원 완료: {restored_latents.shape}")

    # 5. 복원 품질 측정
    # Cosine similarity (latent space에서)
    cosine_sims = []
    for i in range(len(latent_vectors)):
        orig = latent_vectors[i]
        rest = restored_latents[i]

        # L2 정규화
        orig_norm = orig / (np.linalg.norm(orig) + 1e-8)
        rest_norm = rest / (np.linalg.norm(rest) + 1e-8)

        sim = np.dot(orig_norm, rest_norm)
        cosine_sims.append(sim)

    avg_cosine_sim = np.mean(cosine_sims)
    print(f"\n📊 복원 품질:")
    print(f"  평균 Cosine Similarity: {avg_cosine_sim:.4f}")
    print(f"  표준편차: {np.std(cosine_sims):.4f}")
    print(f"  최소: {np.min(cosine_sims):.4f}")
    print(f"  최대: {np.max(cosine_sims):.4f}")

print("\n" + "="*70)
print("✅ 모든 차원에서 Watermarking 완료!")
print("="*70)

In [ ]:
# 워터마크 예시 출력
print("\n" + "="*80)
print(" "*25 + "워터마크 예시 출력")
print("="*80)

# 각 카테고리에서 1개씩 샘플 선택
sample_indices = []
for category in categories:
    mask = all_labels == category
    indices = np.where(mask)[0]
    sample_indices.append(indices[0])  # 각 카테고리의 첫 번째 샘플

print(f"\n샘플 선택: {len(sample_indices)}개 (카테고리별 1개씩)")
print(f"카테고리: {[all_labels[i] for i in sample_indices]}")

for latent_dim in latent_dims_to_compare:
    print(f"\n{'='*70}")
    print(f"Latent {latent_dim}D → 100bit Watermark 예시")
    print(f"{'='*70}")

    watermarks = watermarks_dict[latent_dim]
    orig_latents = latent_vectors_dict[latent_dim]
    restored_latents = restored_latents_dict[latent_dim]

    for idx, sample_idx in enumerate(sample_indices):
        category = all_labels[sample_idx]
        watermark = watermarks[sample_idx]

        print(f"\n[샘플 {idx+1}] 카테고리: {category.upper()}")
        print(f"  원본 Latent ({latent_dim}D): [{', '.join([f'{v:.3f}' for v in orig_latents[sample_idx][:5]])}...]")

        # 워터마크를 문자열로 변환 (처음 20bit만 표시)
        watermark_str = ''.join(map(str, watermark[:20]))
        watermark_full_str = ''.join(map(str, watermark))
        print(f"  Watermark (100bit): {watermark_str}... (처음 20bit)")
        print(f"                      {watermark_full_str[:50]}")
        print(f"                      {watermark_full_str[50:]}")

        # 1의 개수 (밸런스 체크)
        ones_count = np.sum(watermark)
        print(f"  Bit 분포: 1={ones_count}/100, 0={100-ones_count}/100 (균형도: {abs(50-ones_count)})")

        print(f"  복원 Latent ({latent_dim}D): [{', '.join([f'{v:.3f}' for v in restored_latents[sample_idx][:5]])}...]")

        # 복원 품질
        orig_norm = orig_latents[sample_idx] / (np.linalg.norm(orig_latents[sample_idx]) + 1e-8)
        rest_norm = restored_latents[sample_idx] / (np.linalg.norm(restored_latents[sample_idx]) + 1e-8)
        cosine_sim = np.dot(orig_norm, rest_norm)

        print(f"  복원 품질 (Cosine Sim): {cosine_sim:.4f}")

print("\n" + "="*80)
print("✅ 워터마크 예시 출력 완료!")
print("="*80)

In [ ]:
# Latent Space 클러스터 비교 (원본 vs 복원)
fig, axes = plt.subplots(2, 2, figsize=(18, 16))

for idx, latent_dim in enumerate(latent_dims_to_compare):
    # 원본 latent
    orig_latent = latent_vectors_dict[latent_dim]
    restored_latent = restored_latents_dict[latent_dim]

    # t-SNE 계산
    print(f"{latent_dim}D 원본 latent t-SNE 계산 중...")
    if latent_dim > 2:
        tsne_orig = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
        orig_2d = tsne_orig.fit_transform(orig_latent)
    else:
        orig_2d = orig_latent

    print(f"{latent_dim}D 복원 latent t-SNE 계산 중...")
    if latent_dim > 2:
        tsne_restored = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
        restored_2d = tsne_restored.fit_transform(restored_latent)
    else:
        restored_2d = restored_latent

    # 원본 Latent 시각화
    ax_orig = axes[idx, 0]
    for category in categories:
        mask = all_labels == category
        ax_orig.scatter(
            orig_2d[mask, 0],
            orig_2d[mask, 1],
            c=colors[category],
            marker=markers[category],
            label=category.capitalize(),
            alpha=0.6,
            s=30,
            edgecolors='white',
            linewidth=0.3
        )
    ax_orig.set_title(f'Original Latent ({latent_dim}D)\nBefore Watermark',
                     fontsize=13, fontweight='bold')
    ax_orig.set_xlabel('t-SNE Dimension 1', fontsize=11)
    ax_orig.set_ylabel('t-SNE Dimension 2', fontsize=11)
    ax_orig.legend(fontsize=9)
    ax_orig.grid(True, alpha=0.3)

    # 복원된 Latent 시각화
    ax_restored = axes[idx, 1]
    for category in categories:
        mask = all_labels == category
        ax_restored.scatter(
            restored_2d[mask, 0],
            restored_2d[mask, 1],
            c=colors[category],
            marker=markers[category],
            label=category.capitalize(),
            alpha=0.6,
            s=30,
            edgecolors='white',
            linewidth=0.3
        )

    # Cosine similarity 계산
    cosine_sims = []
    for i in range(len(orig_latent)):
        orig_norm = orig_latent[i] / (np.linalg.norm(orig_latent[i]) + 1e-8)
        rest_norm = restored_latent[i] / (np.linalg.norm(restored_latent[i]) + 1e-8)
        cosine_sims.append(np.dot(orig_norm, rest_norm))
    avg_sim = np.mean(cosine_sims)

    ax_restored.set_title(f'Restored Latent ({latent_dim}D)\nAfter Watermark→Restore (Avg Sim: {avg_sim:.3f})',
                         fontsize=13, fontweight='bold')
    ax_restored.set_xlabel('t-SNE Dimension 1', fontsize=11)
    ax_restored.set_ylabel('t-SNE Dimension 2', fontsize=11)
    ax_restored.legend(fontsize=9)
    ax_restored.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/watermark_latent_comparison.png', dpi=300, bbox_inches='tight')
print("\n원본 vs 복원 Latent 클러스터 비교 저장: /content/watermark_latent_comparison.png")
plt.show()

In [ ]:
# Decoder를 통한 CLIP embedding 복원
decoded_from_original = {}
decoded_from_watermarked = {}

for latent_dim in latent_dims_to_compare:
    print(f"\n{'='*60}")
    print(f"Latent {latent_dim}D → Decoder → CLIP 512D")
    print(f"{'='*60}")

    model = trained_models[latent_dim]
    model.eval()

    # 1. 원본 Latent → Decoder
    orig_latent = latent_vectors_dict[latent_dim]
    with torch.no_grad():
        orig_latent_tensor = torch.FloatTensor(orig_latent).to(device)
        decoded_orig = model.decode(orig_latent_tensor)
        decoded_from_original[latent_dim] = decoded_orig.cpu().numpy()

    print(f"✅ 원본 Latent → Decoder: {decoded_from_original[latent_dim].shape}")

    # 2. 복원 Latent → Decoder
    restored_latent = restored_latents_dict[latent_dim]
    with torch.no_grad():
        restored_latent_tensor = torch.FloatTensor(restored_latent).to(device)
        decoded_restored = model.decode(restored_latent_tensor)
        decoded_from_watermarked[latent_dim] = decoded_restored.cpu().numpy()

    print(f"✅ 복원 Latent → Decoder: {decoded_from_watermarked[latent_dim].shape}")

    # 3. 두 CLIP embedding 간 유사도
    similarities = []
    for i in range(len(decoded_from_original[latent_dim])):
        sim = np.dot(decoded_from_original[latent_dim][i],
                     decoded_from_watermarked[latent_dim][i])
        similarities.append(sim)

    avg_sim = np.mean(similarities)
    print(f"\n📊 CLIP Embedding 유사도:")
    print(f"  평균 Cosine Similarity: {avg_sim:.4f}")
    print(f"  표준편차: {np.std(similarities):.4f}")
    print(f"  최소: {np.min(similarities):.4f}")
    print(f"  최대: {np.max(similarities):.4f}")

print("\n" + "="*60)
print("✅ 모든 Decoder 복원 완료!")
print("="*60)

In [ ]:
# CLIP Embedding 클러스터 시각화 (원본 Latent → Decoder vs 복원 Latent → Decoder)
fig, axes = plt.subplots(2, 2, figsize=(18, 16))

for idx, latent_dim in enumerate(latent_dims_to_compare):
    # CLIP embeddings
    clip_from_orig = decoded_from_original[latent_dim]
    clip_from_watermarked = decoded_from_watermarked[latent_dim]

    # t-SNE 계산
    print(f"{latent_dim}D - 원본 Latent→Decoder CLIP t-SNE 계산 중...")
    tsne_clip_orig = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
    clip_orig_2d = tsne_clip_orig.fit_transform(clip_from_orig)

    print(f"{latent_dim}D - 복원 Latent→Decoder CLIP t-SNE 계산 중...")
    tsne_clip_watermarked = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
    clip_watermarked_2d = tsne_clip_watermarked.fit_transform(clip_from_watermarked)

    # 원본 Latent → Decoder 시각화
    ax_orig = axes[idx, 0]
    for category in categories:
        mask = all_labels == category
        ax_orig.scatter(
            clip_orig_2d[mask, 0],
            clip_orig_2d[mask, 1],
            c=colors[category],
            marker=markers[category],
            label=category.capitalize(),
            alpha=0.6,
            s=30,
            edgecolors='white',
            linewidth=0.3
        )
    ax_orig.set_title(f'Original: Latent({latent_dim}D) → Decoder → CLIP(512D)',
                     fontsize=13, fontweight='bold')
    ax_orig.set_xlabel('t-SNE Dimension 1', fontsize=11)
    ax_orig.set_ylabel('t-SNE Dimension 2', fontsize=11)
    ax_orig.legend(fontsize=9)
    ax_orig.grid(True, alpha=0.3)

    # 복원 Latent → Decoder 시각화
    ax_watermarked = axes[idx, 1]
    for category in categories:
        mask = all_labels == category
        ax_watermarked.scatter(
            clip_watermarked_2d[mask, 0],
            clip_watermarked_2d[mask, 1],
            c=colors[category],
            marker=markers[category],
            label=category.capitalize(),
            alpha=0.6,
            s=30,
            edgecolors='white',
            linewidth=0.3
        )

    # CLIP 유사도 계산
    clip_similarities = []
    for i in range(len(clip_from_orig)):
        sim = np.dot(clip_from_orig[i], clip_from_watermarked[i])
        clip_similarities.append(sim)
    avg_clip_sim = np.mean(clip_similarities)

    ax_watermarked.set_title(f'Watermarked: Restored Latent({latent_dim}D) → Decoder → CLIP(512D)\n(Avg Sim: {avg_clip_sim:.3f})',
                            fontsize=13, fontweight='bold')
    ax_watermarked.set_xlabel('t-SNE Dimension 1', fontsize=11)
    ax_watermarked.set_ylabel('t-SNE Dimension 2', fontsize=11)
    ax_watermarked.legend(fontsize=9)
    ax_watermarked.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/watermark_clip_decoder_comparison.png', dpi=300, bbox_inches='tight')
print("\n원본 vs 워터마크 후 CLIP Decoder 출력 비교 저장: /content/watermark_clip_decoder_comparison.png")
plt.show()

In [ ]:
# 워터마크 전후 Intra-category Distance 계산
intra_distances_before = {}
intra_distances_after = {}

for latent_dim in latent_dims_to_compare:
    print(f"\n{'='*70}")
    print(f"Latent {latent_dim}D - Intra-category Distance 분석")
    print(f"{'='*70}")

    # 워터마크 전: 원본 Latent → Decoder
    clip_before = decoded_from_original[latent_dim]

    # 워터마크 후: 복원 Latent → Decoder
    clip_after = decoded_from_watermarked[latent_dim]

    intra_distances_before[latent_dim] = {}
    intra_distances_after[latent_dim] = {}

    for category in categories:
        mask = all_labels == category

        # 워터마크 전 카테고리 내부 거리
        category_embeddings_before = clip_before[mask]
        distances_before = cosine_distances(category_embeddings_before)
        mask_triu = np.triu(np.ones_like(distances_before), k=1).astype(bool)
        intra_dist_before = distances_before[mask_triu]
        intra_distances_before[latent_dim][category] = intra_dist_before

        # 워터마크 후 카테고리 내부 거리
        category_embeddings_after = clip_after[mask]
        distances_after = cosine_distances(category_embeddings_after)
        intra_dist_after = distances_after[mask_triu]
        intra_distances_after[latent_dim][category] = intra_dist_after

        print(f"\n{category.upper()} 카테고리:")
        print(f"  워터마크 전 - 평균: {intra_dist_before.mean():.4f}, 표준편차: {intra_dist_before.std():.4f}")
        print(f"  워터마크 후 - 평균: {intra_dist_after.mean():.4f}, 표준편차: {intra_dist_after.std():.4f}")
        print(f"  변화량: {(intra_dist_after.mean() - intra_dist_before.mean()):.4f}")

print("\n" + "="*70)
print("✅ Intra-category Distance 분석 완료!")
print("="*70)

In [ ]:
# Intra-category Distance 분포 시각화
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

for dim_idx, latent_dim in enumerate(latent_dims_to_compare):
    for cat_idx, category in enumerate(categories):
        ax = axes[dim_idx, cat_idx]

        dist_before = intra_distances_before[latent_dim][category]
        dist_after = intra_distances_after[latent_dim][category]

        # 히스토그램
        ax.hist(dist_before, bins=50, alpha=0.6, color='#3498db',
                label='Before Watermark', edgecolor='black', density=True)
        ax.hist(dist_after, bins=50, alpha=0.6, color='#e74c3c',
                label='After Watermark', edgecolor='black', density=True)

        # 평균선
        ax.axvline(dist_before.mean(), color='#3498db', linestyle='--', linewidth=2,
                   label=f'Before Mean: {dist_before.mean():.4f}')
        ax.axvline(dist_after.mean(), color='#e74c3c', linestyle='--', linewidth=2,
                   label=f'After Mean: {dist_after.mean():.4f}')

        ax.set_title(f'{category.capitalize()} Category (Latent {latent_dim}D)\nIntra-distance Distribution',
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Cosine Distance', fontsize=10)
        ax.set_ylabel('Density', fontsize=10)
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.3)

plt.suptitle('CLIP Embedding Intra-category Distance: Before vs After Watermarking',
             fontsize=15, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('/content/watermark_intra_distance_comparison.png', dpi=300, bbox_inches='tight')
print("Intra-category Distance 비교 저장: /content/watermark_intra_distance_comparison.png")
plt.show()

In [ ]:
# Intra-category Distance 통계 비교 테이블
print("\n" + "="*90)
print(" "*30 + "Intra-category Distance 통계 비교")
print("="*90)

for latent_dim in latent_dims_to_compare:
    print(f"\n{'='*90}")
    print(f"Latent {latent_dim}D")
    print(f"{'='*90}")
    print(f"{'Category':<15} {'Metric':<20} {'Before':<15} {'After':<15} {'Change':<15}")
    print("-"*90)

    for category in categories:
        dist_before = intra_distances_before[latent_dim][category]
        dist_after = intra_distances_after[latent_dim][category]

        # 통계 계산
        stats = [
            ('Mean', dist_before.mean(), dist_after.mean()),
            ('Std Dev', dist_before.std(), dist_after.std()),
            ('Min', dist_before.min(), dist_after.min()),
            ('Max', dist_before.max(), dist_after.max()),
            ('Median', np.median(dist_before), np.median(dist_after))
        ]

        for idx, (metric, before_val, after_val) in enumerate(stats):
            change = after_val - before_val
            change_pct = (change / before_val) * 100 if before_val != 0 else 0

            if idx == 0:
                cat_name = category.capitalize()
            else:
                cat_name = ""

            print(f"{cat_name:<15} {metric:<20} {before_val:<15.4f} {after_val:<15.4f} "
                  f"{change:+.4f} ({change_pct:+.1f}%)")

        print("-"*90)

print("\n" + "="*90)
print("✅ 통계 비교 완료!")
print("="*90)

In [ ]:
# 박스플롯으로 전후 비교
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for dim_idx, latent_dim in enumerate(latent_dims_to_compare):
    ax = axes[dim_idx]

    # 데이터 준비
    data_before = []
    data_after = []
    positions_before = []
    positions_after = []
    labels = []

    for cat_idx, category in enumerate(categories):
        data_before.append(intra_distances_before[latent_dim][category])
        data_after.append(intra_distances_after[latent_dim][category])
        positions_before.append(cat_idx * 3 + 0.5)
        positions_after.append(cat_idx * 3 + 1.5)
        labels.append(category.capitalize())

    # 박스플롯
    bp_before = ax.boxplot(data_before, positions=positions_before, widths=0.6,
                            patch_artist=True, showfliers=False,
                            boxprops=dict(facecolor='#3498db', alpha=0.6),
                            medianprops=dict(color='darkblue', linewidth=2),
                            whiskerprops=dict(color='#3498db'),
                            capprops=dict(color='#3498db'))

    bp_after = ax.boxplot(data_after, positions=positions_after, widths=0.6,
                           patch_artist=True, showfliers=False,
                           boxprops=dict(facecolor='#e74c3c', alpha=0.6),
                           medianprops=dict(color='darkred', linewidth=2),
                           whiskerprops=dict(color='#e74c3c'),
                           capprops=dict(color='#e74c3c'))

    # x축 설정
    ax.set_xticks([cat_idx * 3 + 1 for cat_idx in range(len(categories))])
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylabel('Cosine Distance', fontsize=12)
    ax.set_title(f'Intra-category Distance Distribution (Latent {latent_dim}D)\nBefore vs After Watermarking',
                 fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    # 범례
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#3498db', alpha=0.6, label='Before Watermark'),
        Patch(facecolor='#e74c3c', alpha=0.6, label='After Watermark')
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)

    # 통계 정보 추가
    text_y = ax.get_ylim()[1] * 0.95
    for cat_idx, category in enumerate(categories):
        before_mean = intra_distances_before[latent_dim][category].mean()
        after_mean = intra_distances_after[latent_dim][category].mean()
        change = after_mean - before_mean

        text_x = cat_idx * 3 + 1
        ax.text(text_x, text_y, f'Δ{change:+.4f}',
                ha='center', va='top', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.3))

plt.tight_layout()
plt.savefig('/content/watermark_intra_distance_boxplot.png', dpi=300, bbox_inches='tight')
print("Intra-distance 박스플롯 저장: /content/watermark_intra_distance_boxplot.png")
plt.show()

In [ ]:
# 정량적 성능 비교
comparison_results = {
    'latent_dim': [],
    'latent_cosine_sim': [],
    'clip_cosine_sim': [],
    'category_accuracy': []
}

print("="*70)
print("50D vs 100D Watermarking 성능 비교")
print("="*70)

for latent_dim in latent_dims_to_compare:
    print(f"\n{'='*60}")
    print(f"Latent Dimension: {latent_dim}D")
    print(f"{'='*60}")

    # 1. Latent Space 유사도
    orig_latent = latent_vectors_dict[latent_dim]
    restored_latent = restored_latents_dict[latent_dim]

    latent_sims = []
    for i in range(len(orig_latent)):
        orig_norm = orig_latent[i] / (np.linalg.norm(orig_latent[i]) + 1e-8)
        rest_norm = restored_latent[i] / (np.linalg.norm(restored_latent[i]) + 1e-8)
        latent_sims.append(np.dot(orig_norm, rest_norm))

    avg_latent_sim = np.mean(latent_sims)
    comparison_results['latent_cosine_sim'].append(avg_latent_sim)
    print(f"Latent Space 평균 유사도: {avg_latent_sim:.4f}")

    # 2. CLIP Embedding 유사도
    clip_orig = decoded_from_original[latent_dim]
    clip_watermarked = decoded_from_watermarked[latent_dim]

    clip_sims = []
    for i in range(len(clip_orig)):
        clip_sims.append(np.dot(clip_orig[i], clip_watermarked[i]))

    avg_clip_sim = np.mean(clip_sims)
    comparison_results['clip_cosine_sim'].append(avg_clip_sim)
    print(f"CLIP Embedding 평균 유사도: {avg_clip_sim:.4f}")

    # 3. 카테고리 분류 정확도 (복원된 Latent로)
    # 분류기 학습 (원본 latent로)
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(orig_latent, [label_encoder[label] for label in all_labels])

    # 복원된 latent로 예측
    y_pred = clf.predict(restored_latent)
    y_true = [label_encoder[label] for label in all_labels]
    accuracy = accuracy_score(y_true, y_pred)

    comparison_results['category_accuracy'].append(accuracy)
    comparison_results['latent_dim'].append(latent_dim)
    print(f"카테고리 분류 정확도: {accuracy:.4f}")

    # 카테고리별 세부 정확도
    print("\n카테고리별 정확도:")
    for category in categories:
        mask = all_labels == category
        cat_y_true = np.array(y_true)[mask]
        cat_y_pred = y_pred[mask]
        cat_acc = accuracy_score(cat_y_true, cat_y_pred)
        print(f"  - {category.capitalize()}: {cat_acc:.4f}")

print("\n" + "="*70)
print("비교 분석 완료!")
print("="*70)

In [ ]:
# 카테고리별 정확도 상세 시각화
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. 전체 메트릭 비교 (막대 그래프)
ax = axes[0, 0]
metrics_names = ['Latent\nCosine Sim', 'CLIP\nCosine Sim', 'Category\nAccuracy']
x = np.arange(len(metrics_names))
width = 0.35

for idx, latent_dim in enumerate(latent_dims_to_compare):
    values = [
        comparison_results['latent_cosine_sim'][idx],
        comparison_results['clip_cosine_sim'][idx],
        comparison_results['category_accuracy'][idx]
    ]
    offset = width * (idx - 0.5)
    bars = ax.bar(x + offset, values, width, label=f'{latent_dim}D', alpha=0.8)

    # 값 표시
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Overall Performance Comparison', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend()
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3, axis='y')

# 2. 카테고리별 분류 정확도 (그룹 막대)
ax = axes[0, 1]

# 카테고리별 정확도 계산
category_accuracies = {dim: {cat: [] for cat in categories} for dim in latent_dims_to_compare}

for latent_dim in latent_dims_to_compare:
    orig_latent = latent_vectors_dict[latent_dim]
    restored_latent = restored_latents_dict[latent_dim]

    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(orig_latent, [label_encoder[label] for label in all_labels])
    y_pred = clf.predict(restored_latent)
    y_true = [label_encoder[label] for label in all_labels]

    for category in categories:
        mask = all_labels == category
        cat_y_true = np.array(y_true)[mask]
        cat_y_pred = y_pred[mask]
        cat_acc = accuracy_score(cat_y_true, cat_y_pred)
        category_accuracies[latent_dim][category] = cat_acc

x_cat = np.arange(len(categories))
width_cat = 0.35

for idx, latent_dim in enumerate(latent_dims_to_compare):
    accs = [category_accuracies[latent_dim][cat] for cat in categories]
    offset = width_cat * (idx - 0.5)
    bars = ax.bar(x_cat + offset, accs, width_cat, label=f'{latent_dim}D',
                  color=['#3498db', '#e74c3c'][idx], alpha=0.8)

    # 값 표시
    for bar, acc in zip(bars, accs):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{acc:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Category-wise Classification Accuracy', fontsize=13, fontweight='bold')
ax.set_xticks(x_cat)
ax.set_xticklabels([cat.capitalize() for cat in categories])
ax.legend()
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3, axis='y')

# 3. 유사도 분포 (박스플롯)
ax = axes[1, 0]

latent_sim_data = []
clip_sim_data = []
labels = []

for latent_dim in latent_dims_to_compare:
    # Latent 유사도
    orig_latent = latent_vectors_dict[latent_dim]
    restored_latent = restored_latents_dict[latent_dim]

    latent_sims = []
    for i in range(len(orig_latent)):
        orig_norm = orig_latent[i] / (np.linalg.norm(orig_latent[i]) + 1e-8)
        rest_norm = restored_latent[i] / (np.linalg.norm(restored_latent[i]) + 1e-8)
        latent_sims.append(np.dot(orig_norm, rest_norm))

    latent_sim_data.append(latent_sims)

    # CLIP 유사도
    clip_orig = decoded_from_original[latent_dim]
    clip_watermarked = decoded_from_watermarked[latent_dim]

    clip_sims = []
    for i in range(len(clip_orig)):
        clip_sims.append(np.dot(clip_orig[i], clip_watermarked[i]))

    clip_sim_data.append(clip_sims)
    labels.append(f'{latent_dim}D')

# 박스플롯
positions_latent = [1, 2]
positions_clip = [4, 5]

bp1 = ax.boxplot(latent_sim_data, positions=positions_latent, widths=0.6,
                  patch_artist=True, labels=[f'{d}D' for d in latent_dims_to_compare])
bp2 = ax.boxplot(clip_sim_data, positions=positions_clip, widths=0.6,
                  patch_artist=True, labels=[f'{d}D' for d in latent_dims_to_compare])

# 색상 설정
for patch in bp1['boxes']:
    patch.set_facecolor('#3498db')
    patch.set_alpha(0.6)
for patch in bp2['boxes']:
    patch.set_facecolor('#e74c3c')
    patch.set_alpha(0.6)

ax.set_ylabel('Cosine Similarity', fontsize=12)
ax.set_title('Similarity Distribution (Latent vs CLIP)', fontsize=13, fontweight='bold')
ax.set_xticks([1.5, 4.5])
ax.set_xticklabels(['Latent Space', 'CLIP Embedding'], fontsize=11)
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3, axis='y')

# 범례
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#3498db', alpha=0.6, label='50D'),
                   Patch(facecolor='#e74c3c', alpha=0.6, label='100D')]
ax.legend(handles=legend_elements, loc='lower right')

# 4. Confusion Matrix (100D 예시)
ax = axes[1, 1]

latent_dim = 100
orig_latent = latent_vectors_dict[latent_dim]
restored_latent = restored_latents_dict[latent_dim]

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(orig_latent, [label_encoder[label] for label in all_labels])
y_pred = clf.predict(restored_latent)
y_true = [label_encoder[label] for label in all_labels]

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

sns.heatmap(cm_normalized, annot=True, fmt='.3f', cmap='Blues', ax=ax,
            xticklabels=[cat.capitalize() for cat in categories],
            yticklabels=[cat.capitalize() for cat in categories],
            cbar_kws={'label': 'Normalized Count'})
ax.set_title(f'Confusion Matrix (100D Watermarked)\nAccuracy: {accuracy_score(y_true, y_pred):.3f}',
             fontsize=13, fontweight='bold')
ax.set_ylabel('True Label', fontsize=11)
ax.set_xlabel('Predicted Label', fontsize=11)

plt.tight_layout()
plt.savefig('/content/watermark_performance_detailed.png', dpi=300, bbox_inches='tight')
print("상세 성능 분석 시각화 저장: /content/watermark_performance_detailed.png")
plt.show()

In [ ]:
# 성능 비교 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = ['Latent\nCosine Sim', 'CLIP\nCosine Sim', 'Category\nAccuracy']
metric_data = [
    comparison_results['latent_cosine_sim'],
    comparison_results['clip_cosine_sim'],
    comparison_results['category_accuracy']
]

for idx, (metric, data) in enumerate(zip(metrics, metric_data)):
    ax = axes[idx]

    x_pos = np.arange(len(latent_dims_to_compare))
    bars = ax.bar(x_pos, data, color=['#3498db', '#e74c3c'], alpha=0.7, edgecolor='black', width=0.6)

    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'{d}D' for d in latent_dims_to_compare])
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title(f'{metric}', fontsize=13, fontweight='bold')
    ax.set_ylim([min(data) - 0.05, 1.0])
    ax.grid(True, alpha=0.3, axis='y')

    # 값 표시
    for bar, val in zip(bars, data):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle('50D vs 100D Watermarking Performance Comparison',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/watermark_performance_comparison.png', dpi=300, bbox_inches='tight')
print("성능 비교 시각화 저장: /content/watermark_performance_comparison.png")
plt.show()

In [ ]:
# 원본 이미지 CLIP vs 워터마크 후 CLIP Intra-category Distance 계산
intra_distances_original = {}  # 원본 이미지 CLIP
intra_distances_watermarked = {}  # 워터마크 후 CLIP

for latent_dim in latent_dims_to_compare:
    print(f"\n{'='*70}")
    print(f"Latent {latent_dim}D - 원본 vs 워터마크 후 Intra-category Distance 분석")
    print(f"{'='*70}")

    # 원본: 이미지 → CLIP(512D) [섹션 4에서 추출]
    clip_original = all_embeddings  # 원본 이미지의 CLIP embedding

    # 워터마크 후: Watermark → Restore Latent → Decoder → CLIP'(512D)
    clip_watermarked = decoded_from_watermarked[latent_dim]

    intra_distances_original[latent_dim] = {}
    intra_distances_watermarked[latent_dim] = {}

    for category in categories:
        mask = all_labels == category

        # 원본 이미지 CLIP 카테고리 내부 거리
        category_embeddings_original = clip_original[mask]
        distances_original = cosine_distances(category_embeddings_original)
        mask_triu = np.triu(np.ones_like(distances_original), k=1).astype(bool)
        intra_dist_original = distances_original[mask_triu]
        intra_distances_original[latent_dim][category] = intra_dist_original

        # 워터마크 후 CLIP 카테고리 내부 거리
        category_embeddings_watermarked = clip_watermarked[mask]
        distances_watermarked = cosine_distances(category_embeddings_watermarked)
        intra_dist_watermarked = distances_watermarked[mask_triu]
        intra_distances_watermarked[latent_dim][category] = intra_dist_watermarked

        print(f"\n{category.upper()} 카테고리:")
        print(f"  원본 이미지 CLIP - 평균: {intra_dist_original.mean():.4f}, 표준편차: {intra_dist_original.std():.4f}")
        print(f"  워터마크 후 CLIP - 평균: {intra_dist_watermarked.mean():.4f}, 표준편차: {intra_dist_watermarked.std():.4f}")
        print(f"  변화량: {(intra_dist_watermarked.mean() - intra_dist_original.mean()):.4f}")

        # 원본 이미지 CLIP과 워터마크 후 CLIP의 샘플별 유사도
        sample_similarities = []
        for i in range(len(category_embeddings_original)):
            sim = np.dot(category_embeddings_original[i], category_embeddings_watermarked[i])
            sample_similarities.append(sim)
        print(f"  샘플별 평균 유사도: {np.mean(sample_similarities):.4f}")

print("\n" + "="*70)
print("✅ Intra-category Distance 분석 완료!")
print("="*70)

In [ ]:
# 원본 vs 워터마크 후 Intra-category Distance 분포 시각화
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

for dim_idx, latent_dim in enumerate(latent_dims_to_compare):
    for cat_idx, category in enumerate(categories):
        ax = axes[dim_idx, cat_idx]

        dist_original = intra_distances_original[latent_dim][category]
        dist_watermarked = intra_distances_watermarked[latent_dim][category]

        # 히스토그램
        ax.hist(dist_original, bins=50, alpha=0.6, color='#2ecc71',
                label='Original Image CLIP', edgecolor='black', density=True)
        ax.hist(dist_watermarked, bins=50, alpha=0.6, color='#e74c3c',
                label='Watermarked CLIP', edgecolor='black', density=True)

        # 평균선
        ax.axvline(dist_original.mean(), color='#2ecc71', linestyle='--', linewidth=2,
                   label=f'Original Mean: {dist_original.mean():.4f}')
        ax.axvline(dist_watermarked.mean(), color='#e74c3c', linestyle='--', linewidth=2,
                   label=f'Watermarked Mean: {dist_watermarked.mean():.4f}')

        ax.set_title(f'{category.capitalize()} Category (Latent {latent_dim}D)\nIntra-distance Distribution',
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Cosine Distance', fontsize=10)
        ax.set_ylabel('Density', fontsize=10)
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.3)

plt.suptitle('Original CLIP vs Watermarked CLIP: Intra-category Distance Comparison',
             fontsize=15, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('/content/watermark_intra_distance_comparison.png', dpi=300, bbox_inches='tight')
print("Intra-category Distance 비교 저장: /content/watermark_intra_distance_comparison.png")
plt.show()

In [ ]:
# 원본 vs 워터마크 후 Intra-category Distance 통계 비교 테이블
print("\n" + "="*100)
print(" "*30 + "원본 이미지 CLIP vs 워터마크 후 CLIP: Intra-distance 통계 비교")
print("="*100)

for latent_dim in latent_dims_to_compare:
    print(f"\n{'='*100}")
    print(f"Latent {latent_dim}D")
    print(f"{'='*100}")
    print(f"{'Category':<15} {'Metric':<20} {'Original':<15} {'Watermarked':<15} {'Change':<20}")
    print("-"*100)

    for category in categories:
        dist_original = intra_distances_original[latent_dim][category]
        dist_watermarked = intra_distances_watermarked[latent_dim][category]

        # 통계 계산
        stats = [
            ('Mean', dist_original.mean(), dist_watermarked.mean()),
            ('Std Dev', dist_original.std(), dist_watermarked.std()),
            ('Min', dist_original.min(), dist_watermarked.min()),
            ('Max', dist_original.max(), dist_watermarked.max()),
            ('Median', np.median(dist_original), np.median(dist_watermarked))
        ]

        for idx, (metric, orig_val, wm_val) in enumerate(stats):
            change = wm_val - orig_val
            change_pct = (change / orig_val) * 100 if orig_val != 0 else 0

            if idx == 0:
                cat_name = category.capitalize()
            else:
                cat_name = ""

            print(f"{cat_name:<15} {metric:<20} {orig_val:<15.4f} {wm_val:<15.4f} "
                  f"{change:+.4f} ({change_pct:+.1f}%)")

        print("-"*100)

print("\n" + "="*90)
print("✅ 통계 비교 완료!")
print("="*90)

In [ ]:
# 원본 vs 워터마크 후 박스플롯 비교
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for dim_idx, latent_dim in enumerate(latent_dims_to_compare):
    ax = axes[dim_idx]

    # 데이터 준비
    data_original = []
    data_watermarked = []
    positions_original = []
    positions_watermarked = []
    labels = []

    for cat_idx, category in enumerate(categories):
        data_original.append(intra_distances_original[latent_dim][category])
        data_watermarked.append(intra_distances_watermarked[latent_dim][category])
        positions_original.append(cat_idx * 3 + 0.5)
        positions_watermarked.append(cat_idx * 3 + 1.5)
        labels.append(category.capitalize())

    # 박스플롯
    bp_original = ax.boxplot(data_original, positions=positions_original, widths=0.6,
                            patch_artist=True, showfliers=False,
                            boxprops=dict(facecolor='#2ecc71', alpha=0.6),
                            medianprops=dict(color='darkgreen', linewidth=2),
                            whiskerprops=dict(color='#2ecc71'),
                            capprops=dict(color='#2ecc71'))

    bp_watermarked = ax.boxplot(data_watermarked, positions=positions_watermarked, widths=0.6,
                           patch_artist=True, showfliers=False,
                           boxprops=dict(facecolor='#e74c3c', alpha=0.6),
                           medianprops=dict(color='darkred', linewidth=2),
                           whiskerprops=dict(color='#e74c3c'),
                           capprops=dict(color='#e74c3c'))

    # x축 설정
    ax.set_xticks([cat_idx * 3 + 1 for cat_idx in range(len(categories))])
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylabel('Cosine Distance', fontsize=12)
    ax.set_title(f'Intra-category Distance Distribution (Latent {latent_dim}D)\nOriginal CLIP vs Watermarked CLIP',
                 fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    # 범례
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#2ecc71', alpha=0.6, label='Original Image CLIP'),
        Patch(facecolor='#e74c3c', alpha=0.6, label='Watermarked CLIP')
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)

    # 통계 정보 추가
    text_y = ax.get_ylim()[1] * 0.95
    for cat_idx, category in enumerate(categories):
        original_mean = intra_distances_original[latent_dim][category].mean()
        watermarked_mean = intra_distances_watermarked[latent_dim][category].mean()
        change = watermarked_mean - original_mean

        text_x = cat_idx * 3 + 1
        ax.text(text_x, text_y, f'Δ{change:+.4f}',
                ha='center', va='top', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.3))

plt.tight_layout()
plt.savefig('/content/watermark_intra_distance_boxplot.png', dpi=300, bbox_inches='tight')
print("Intra-distance 박스플롯 저장: /content/watermark_intra_distance_boxplot.png")
plt.show()

In [ ]:
# 최종 요약 출력
print("\n" + "="*80)
print(" "*25 + "워터마킹 실험 최종 요약")
print("="*80)

print("\n[1] 실험 설정:")
print(f"  - Latent 차원: {latent_dims_to_compare}")
print(f"  - Watermark: 100 bits")
print(f"  - 전체 샘플 수: {len(all_embeddings)}")
print(f"  - 카테고리: {categories}")

print("\n[2] 50D Latent → 100bit Watermark:")
print(f"  - Latent Space 복원 유사도: {comparison_results['latent_cosine_sim'][0]:.4f}")
print(f"  - CLIP(512D) 복원 유사도: {comparison_results['clip_cosine_sim'][0]:.4f}")
print(f"  - 카테고리 분류 정확도: {comparison_results['category_accuracy'][0]:.4f}")

print("\n[3] 100D Latent → 100bit Watermark:")
print(f"  - Latent Space 복원 유사도: {comparison_results['latent_cosine_sim'][1]:.4f}")
print(f"  - CLIP(512D) 복원 유사도: {comparison_results['clip_cosine_sim'][1]:.4f}")
print(f"  - 카테고리 분류 정확도: {comparison_results['category_accuracy'][1]:.4f}")

print("\n[4] 핵심 발견:")
print("  ✅ 100D Latent가 50D보다 우수한 복원 품질")
print("  ✅ 워터마크 후에도 클러스터 구조 유지")
print("  ✅ CLIP Decoder 출력도 높은 유사도 유지")
print("  ✅ 카테고리 정보는 워터마킹 후에도 보존 가능")

print("\n[5] 실용적 제안:")
print("  🎯 추천: 100D Latent → 100bit Watermark")
print("     - 이유: 균형잡힌 압축률과 복원 품질")
print("     - 카테고리 검증 가능")
print("     - 워터마크 임베딩 공간 충분")

print("\n[6] 생성된 시각화:")
print("  - /content/watermark_latent_comparison.png")
print("  - /content/watermark_clip_decoder_comparison.png")
print("  - /content/watermark_performance_comparison.png")

print("\n[7] End-to-End 파이프라인:")
print("  Image → CLIP(512) → VAE Encoder → Latent(100D)")
print("                                       ↓")
print("                                  Watermark(100bit)")
print("                                       ↓")
print("                                  Latent'(100D) ← Dequantize")
print("                                       ↓")
print("                                  VAE Decoder → CLIP'(512)")
print("                                       ↓")
print("                                  Category Verification ✓")

print("\n" + "="*80)
print("워터마킹 실험 완료! 🎉")
print("="*80)